# Docker + Kubernetes — First Contact

A **container** packages an application together with its runtime, libraries, and operating-system-level dependencies so it runs the same way everywhere. A **virtual machine** virtualizes an entire operating system; a container shares the host kernel and is therefore lighter, faster to start, and easier to scale. For a data engineer, that difference matters because most modern platforms are made of many small services that need to start consistently and move cleanly across laptops, test environments, and production clusters.

An **image** is the immutable blueprint; a **container** is the running instance created from that blueprint. You build an image once, then run one or many containers from it. This image-to-container model is the foundation for reproducible pipelines, portable tooling, and predictable deployments. Instead of “it works on my machine,” containers give you a packaged unit that behaves the same in more places.

Why does this matter for data engineering? Because the real platform is rarely one big program. It is Kafka brokers, Airflow schedulers, Spark jobs, ML services, databases, observability agents, and supporting tools all running as separate services. **Every service in the Citi telemetry stack — Kafka, Spark, Airflow, MLflow — runs in a Docker container. Understanding containers means understanding your infrastructure.**

```text
[Dockerfile] → [docker build] → [Image] → [docker run] → [Container]

[Container 1] [Container 2] [Container 3] ... → [Docker Compose] → [Stack]
```

In [1]:
import subprocess, json, os, pathlib


## Docker CLI — the 10 commands you use daily

Below, we run a few of the most useful Docker commands directly from Python using `subprocess`. This is a good pattern for notebooks because it keeps the commands executable, inspectable, and easy to automate later.


In [2]:
def run_command(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    return result.returncode, result.stdout.strip(), result.stderr.strip()

print("=== 1) docker version ===")
rc, out, err = run_command(["docker", "version", "--format", "{{json .}}"])
if rc == 0 and out:
    try:
        payload = json.loads(out)
        server_version = payload.get("Server", {}).get("Version", "Unknown")
        print(f"Server Engine version: {server_version}")
    except Exception:
        print(out)
else:
    print(err or "docker version failed")

print("\n=== 2) docker ps ===")
rc, out, err = run_command(["docker", "ps", "--format", "table {{.Names}}\t{{.Status}}\t{{.Ports}}"])
print(out if out else err)

print("\n=== 3) docker images (top 10) ===")
rc, out, err = run_command(["docker", "images", "--format", "table {{.Repository}}\t{{.Tag}}\t{{.Size}}"])
if out:
    lines = out.splitlines()
    print("\n".join(lines[:11]))
else:
    print(err)

print("\n=== 4) docker stats --no-stream ===")
rc, out, err = run_command(["docker", "stats", "--no-stream", "--format", "table {{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}"])
print(out if out else err)

=== 1) docker version ===


Server Engine version: 29.1.3

=== 2) docker ps ===


NAMES              STATUS                  PORTS
citi_splunk        Up 9 hours (healthy)    0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp, 0.0.0.0:8088-8089->8088-8089/tcp, [::]:8088-8089->8088-8089/tcp
citi_kafka_ui      Up 14 hours             0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp
citi_kafka         Up 14 hours             0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp, 0.0.0.0:29092->29092/tcp, [::]:29092->29092/tcp
citi_airflow       Up 14 hours             0.0.0.0:8082->8080/tcp, [::]:8082->8080/tcp
citi_zookeeper     Up 14 hours (healthy)   0.0.0.0:2181->2181/tcp, [::]:2181->2181/tcp
citi_spark         Up 14 hours             0.0.0.0:7077->7077/tcp, [::]:7077->7077/tcp, 0.0.0.0:8081->8080/tcp, [::]:8081->8080/tcp
citi_mlflow        Up 14 hours             0.0.0.0:5000->5000/tcp, [::]:5000->5000/tcp
de_redis           Up 15 hours (healthy)   0.0.0.0:6380->6379/tcp, [::]:6380->6379/tcp
de_kibana          Up 15 hours (healthy)   0.0.0.0:5601->5601/tcp, [::]:5601->5601/tcp
de_neo4j 

REPOSITORY                                                            TAG             SIZE
citi-checker                                                          1.0             235MB
neo4j                                                                 5               988MB
cassandra                                                             4.1             501MB
influxdb                                                              2.7             553MB
postgres                                                              16              641MB
ghcr.io/mlflow/mlflow                                                 latest          1.71GB
postgres                                                              15              633MB
redis                                                                 7-alpine        61.2MB
ghcr.io/openclaw/openclaw                                             latest          3.93GB
qdrant/qdrant                                                         latest  

NAME               CPU %     MEM USAGE / LIMIT
citi_splunk        2.11%     879.4MiB / 31.25GiB
citi_kafka_ui      0.09%     610.4MiB / 31.25GiB
citi_kafka         0.58%     599.1MiB / 31.25GiB
citi_airflow       2.57%     1.466GiB / 31.25GiB
citi_zookeeper     0.14%     224.2MiB / 31.25GiB
citi_spark         0.10%     246.4MiB / 31.25GiB
citi_mlflow        1.15%     1.501GiB / 31.25GiB
de_redis           0.35%     10.51MiB / 31.25GiB
de_kibana          2.99%     693.5MiB / 31.25GiB
de_neo4j           0.58%     969.1MiB / 31.25GiB
de_cassandra       1.37%     1.703GiB / 31.25GiB
de_postgres        0.74%     122.4MiB / 31.25GiB
de_elasticsearch   0.59%     1.826GiB / 31.25GiB
de_influxdb        0.01%     110.3MiB / 31.25GiB


Inspect the `citi_kafka` container — see its configuration, network, mounts.


In [3]:
\
print("=== docker inspect citi_kafka ===")
rc, out, err = run_command(["docker", "inspect", "citi_kafka"])
if rc != 0 or not out:
    print(err or "Unable to inspect citi_kafka")
else:
    try:
        data = json.loads(out)[0]
        image_name = data.get("Config", {}).get("Image")
        status = data.get("State", {}).get("Status")
        networks = data.get("NetworkSettings", {}).get("Networks", {})
        ip_address = "N/A"
        if networks:
            first_network = next(iter(networks.values()))
            ip_address = first_network.get("IPAddress", "N/A")
        ports = data.get("NetworkSettings", {}).get("Ports", {}) or {}
        mounts = data.get("Mounts", []) or []

        print(f"Image: {image_name}")
        print(f"Status: {status}")
        print(f"IPAddress: {ip_address}")

        print("\nPorts mapping:")
        if ports:
            for container_port, host_bindings in ports.items():
                if host_bindings:
                    for binding in host_bindings:
                        print(f"  {container_port} -> {binding.get('HostIp')}:{binding.get('HostPort')}")
                else:
                    print(f"  {container_port} -> not published")
        else:
            print("  No ports found")

        print("\nMounts:")
        if mounts:
            for mount in mounts:
                print(f"  {mount.get('Source')} -> {mount.get('Destination')}")
        else:
            print("  No mounts found")
    except Exception as ex:
        print(f"Failed to parse inspect output: {ex}")

print("\n=== docker logs citi_kafka --tail 5 ===")
rc, out, err = run_command(["docker", "logs", "citi_kafka", "--tail", "5"])
combined = "\n".join([part for part in [out, err] if part]).strip()
print(combined if combined else "No logs returned")


=== docker inspect citi_kafka ===


Image: confluentinc/cp-kafka:7.6.0
Status: running
IPAddress: 172.20.0.6

Ports mapping:
  29092/tcp -> 0.0.0.0:29092
  29092/tcp -> :::29092
  9092/tcp -> 0.0.0.0:9092
  9092/tcp -> :::9092

Mounts:
  /var/lib/docker/volumes/setup_kafka-data/_data -> /var/lib/kafka/data
  /var/lib/docker/volumes/9b308608378798bd06382a6b7ce81032f0db9052a129b1fff75b21860d1d934f/_data -> /etc/kafka/secrets

=== docker logs citi_kafka --tail 5 ===


[2026-04-01 03:27:56,107] TRACE [Controller id=1] Leader imbalance ratio for broker 1 is 0.0 (kafka.controller.KafkaController)
[2026-04-01 03:32:56,094] INFO [Controller id=1] Processing automatic preferred replica leader election (kafka.controller.KafkaController)
[2026-04-01 03:32:56,094] TRACE [Controller id=1] Checking need to trigger auto leader balancing (kafka.controller.KafkaController)
[2026-04-01 03:32:56,094] DEBUG [Controller id=1] Topics not in preferred replica for broker 1 HashMap() (kafka.controller.KafkaController)
[2026-04-01 03:32:56,094] TRACE [Controller id=1] Leader imbalance ratio for broker 1 is 0.0 (kafka.controller.KafkaController)


## Building a Custom Image

A `Dockerfile` is a set of instructions that builds an image layer by layer. Each line adds something: a base image, installed packages, copied code, a working directory, and finally the default command to run when the container starts.


In [4]:
import tempfile

build_dir = pathlib.Path(tempfile.gettempdir()) / "citi_checker"
build_dir.mkdir(parents=True, exist_ok=True)

dockerfile_text = """FROM python:3.11-slim
RUN pip install psycopg2-binary requests
COPY check.py /app/check.py
WORKDIR /app
CMD ["python", "check.py"]
"""

check_py_text = """import psycopg2, json
conn = psycopg2.connect(host="host.docker.internal", port=5432,
    dbname="de_telemetry", user="de_admin", password="DeAdmin2026!")
cur = conn.cursor()
cur.execute("SELECT severity, COUNT(*) FROM public.alerts GROUP BY severity ORDER BY COUNT(*) DESC")
rows = cur.fetchall()
print(json.dumps({row[0]: row[1] for row in rows}, indent=2))
conn.close()
"""

(build_dir / "Dockerfile").write_text(dockerfile_text, encoding="utf-8")
(build_dir / "check.py").write_text(check_py_text, encoding="utf-8")

print("=== docker build -t citi-checker:1.0 ===")
rc, out, err = run_command(["docker", "build", "-t", "citi-checker:1.0", str(build_dir)])
print(out if out else err)

# On Docker Desktop for Windows, host.docker.internal is provided automatically — no --add-host needed
print("\n=== docker run --rm citi-checker:1.0 ===")
rc, out, err = run_command(["docker", "run", "--rm", "citi-checker:1.0"])
print(out if out else err)

=== docker build -t citi-checker:1.0 ===


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 175B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 0.3s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [1/4] FROM docker.io/library/python:3.11-slim@sha256:9358444059ed78e2975ada2c189f1c1a3144a5dab6f35bff8c981afb38946634
#4 resolve docker.io/library/python:3.11-slim@sha256:9358444059ed78e2975ada2c189f1c1a3144a5dab6f35bff8c981afb38946634 0.0s done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 418B done
#5 DONE 0.0s

#6 [2/4] RUN pip install psycopg2-binary requests
#6 CACHED

#7 [3/4] COPY check.py /app/check.py
#7 CACHED

#8 [4/4] WORKDIR /app
#8 CACHED

#9 exporting to image
#9 exporting layers done
#9 exporting manifest sha256:c2ac56e6634e864f1536607706648a396cd4d3a725de433d4d3c93e297b7416b done
#9 exporting config sha256:e2ffcf2a413a

{
  "HIGH": 6313,
  "LOW": 6254,
  "MEDIUM": 6248,
  "CRITICAL": 6185
}


## Docker Compose — Orchestrating Multiple Containers

Docker Compose lets you define and run a **multi-container application** using one YAML file. Instead of starting Kafka, Postgres, Airflow, and supporting services one by one, you describe them as **services** and let Compose start them together. This is how local platform stacks become repeatable.

The main ideas:
- **services**: each named containerized component, such as Kafka, Postgres, or Airflow
- **networks**: shared communication layers so services can reach each other by name
- **volumes**: persistent storage for things like database files or Kafka logs
- **depends_on**: startup dependency hints between services
- **env_file / environment**: inject configuration values
- **health checks**: signal whether a service is actually ready, not just started

A simplified version of a Citi-style local stack might look like this:

```yaml
services:
  postgres:
    image: postgres:16
    container_name: citi_postgres
    env_file: .env
    volumes:
      - pgdata:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U de_admin"]
      interval: 10s
      timeout: 5s
      retries: 5

  kafka:
    image: confluentinc/cp-kafka:latest
    container_name: citi_kafka
    depends_on:
      - postgres
    networks:
      - citi_net

  airflow:
    image: apache/airflow:latest
    container_name: citi_airflow
    depends_on:
      - postgres
      - kafka
    networks:
      - citi_net

networks:
  citi_net:

volumes:
  pgdata:
```

Key insight: **`docker compose up -d` starts all services in dependency order; `docker compose down` removes them.**


## Kubernetes — Container Orchestration at Scale

Docker is great for building and running containers. Kubernetes is the system that manages many containers across one or more machines. Think of a **cluster** as a control plane making decisions and one or more worker nodes doing the actual work. The control plane receives requests, schedules workloads, and keeps the desired state intact.

The smallest deployable unit in Kubernetes is the **Pod**. A Pod usually wraps one container, but it can hold more than one when they must share network identity or storage very closely. In interview language: Docker runs containers; Kubernetes orchestrates them.

Why use Kubernetes instead of only Docker Compose? Because real production systems need **self-healing**, **scaling**, **rolling deployments**, and **multi-node coordination**. Compose is excellent for local development. Kubernetes is what teams use when the platform must survive failures, grow under load, and update safely without stopping the whole system.

```text
[kubectl] → [API Server] → [Scheduler] → [Worker Node]
                                            ├── Pod: citi-kafka
                                            ├── Pod: citi-spark
                                            └── Pod: citi-airflow
```

Note: **Docker Desktop includes a single-node K8s cluster. Enable: Docker Desktop → Settings → Kubernetes → Enable**


Verify `kubectl` is available.

In [5]:
\
print("=== kubectl version --client ===")
try:
    rc, out, err = run_command(["kubectl", "version", "--client"])
    print(out if out else err)
except FileNotFoundError:
    print("Enable Kubernetes in Docker Desktop Settings → Kubernetes → Enable Kubernetes → Apply")

print("\n=== kubectl cluster-info ===")
k8s_available = False
try:
    rc, out, err = run_command(["kubectl", "cluster-info"])
    text = out if out else err
    print(text)
    if rc == 0:
        k8s_available = True
except FileNotFoundError:
    print("Enable Kubernetes in Docker Desktop Settings → Kubernetes → Enable Kubernetes → Apply")


=== kubectl version --client ===
Client Version: v1.34.1
Kustomize Version: v5.7.1

=== kubectl cluster-info ===


To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.


## First Kubernetes Pod

A **Pod** is a wrapper around one or more containers that share a network namespace and can share storage. For a first contact demo, we will run the `citi-checker` image as a single Pod, read its logs, inspect its status, and then delete it.


In [6]:
\
pod_yaml_path = pathlib.Path("/tmp/citi-checker-pod.yaml")
pod_yaml_text = """apiVersion: v1
kind: Pod
metadata:
  name: citi-checker
  labels:
    app: citi-checker
spec:
  containers:
  - name: citi-checker
    image: citi-checker:1.0
    imagePullPolicy: Never
  hostAliases:
  - ip: "host-gateway"
    hostnames:
    - "host.docker.internal"
  restartPolicy: Never
"""
pod_yaml_path.write_text(pod_yaml_text, encoding="utf-8")

if not k8s_available:
    print("Kubernetes is not enabled or cluster-info is unavailable. Enable it in Docker Desktop Settings → Kubernetes → Enable Kubernetes → Apply.")
else:
    print("=== kubectl apply -f /tmp/citi-checker-pod.yaml ===")
    rc, out, err = run_command(["kubectl", "apply", "-f", str(pod_yaml_path)])
    print(out if out else err)

    print("\nWaiting 10 seconds for the Pod to start...")
    import time
    time.sleep(10)

    print("\n=== kubectl logs citi-checker ===")
    rc, out, err = run_command(["kubectl", "logs", "citi-checker"])
    print(out if out else err)

    print("\n=== kubectl get pod citi-checker -o wide ===")
    rc, out, err = run_command(["kubectl", "get", "pod", "citi-checker", "-o", "wide"])
    print(out if out else err)

    print("\n=== kubectl delete pod citi-checker ===")
    rc, out, err = run_command(["kubectl", "delete", "pod", "citi-checker"])
    print(out if out else err)


Kubernetes is not enabled or cluster-info is unavailable. Enable it in Docker Desktop Settings → Kubernetes → Enable Kubernetes → Apply.


## K8s Objects — the 6 you must know for DE interviews

| Object | What it does | DE use case |
|--------|-------------|-------------|
| Pod | Runs one or more containers | A single Spark executor |
| Deployment | Manages N replicas of a Pod, self-healing | Kafka Connect workers |
| Service | Stable network endpoint for Pods | Expose Airflow webserver |
| ConfigMap | Injects non-secret config into Pods | Airflow env vars |
| Secret | Injects secrets (base64) into Pods | DB passwords |
| PersistentVolumeClaim | Attaches durable storage to a Pod | Kafka data volume |


## What Just Happened

- You used Docker CLI commands to inspect the local container runtime.
- You looked inside a running container and checked logs, ports, mounts, and network info.
- You built and ran a custom image that executes a small data-engineering-style database check.
- You saw the Docker Compose mental model for orchestrating a full local stack.
- You verified a Kubernetes cluster (if enabled) and deployed a first Pod.

Citi tie-in: **Every Spark executor in production Citi runs as a K8s Pod. The KubernetesExecutor in Airflow spins a Pod per task and deletes it when done.**

Next: **Run `terraform_intro.ipynb` for IaC, then `infra_concepts.md` for vocabulary.**
